# N05 Transformer 组件消融

> **对应理论篇**：[`03-Attention与Transformer.md`](../../01-基础/03-Attention与Transformer.md)、[`06-初始化与学习率调度.md`](../../01-基础/06-初始化与学习率调度.md)
> **对应 AlphaProof 源码**：`nanoproof/nanoproof/model.py`（GQA、RoPE、MLP、残差）
> **Colab**：CPU 可跑（5 组消融，每组 150 步，约 2–3 分钟）；GPU 更快。

## 学习目标
1. 用控制变量法做组件消融：只改一处，其余固定；
2. 量化 GQA、RoPE、权重共享、MLP 激活各自的影响；
3. 养成「先跑基线，再单变量对比」的实验习惯。


In [1]:
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu 2>/dev/null || pip -q install torch

In [2]:
import os, sys
def find_code_dir():
    for c in ["../../01-基础/code/from_scratch", "01-基础/code/from_scratch",
              "../01-基础/code/from_scratch", "code/from_scratch"]:
        if os.path.exists(os.path.join(c, "configs.py")):
            return os.path.abspath(c)
    cur = os.getcwd()
    for _ in range(8):
        c = os.path.join(cur, "01-基础", "code", "from_scratch", "configs.py")
        if os.path.exists(c):
            return os.path.dirname(c)
        cur = os.path.dirname(cur)
    raise FileNotFoundError("找不到 from_scratch")
CODE_DIR = find_code_dir()
sys.path.insert(0, CODE_DIR)
print("代码目录:", CODE_DIR)


代码目录: /mnt/workspace/alphaproof-learn/stage/v1-stage/01-基础/code/from_scratch


In [3]:
import copy, math, torch
import torch.nn as nn
import model as M
from configs import get_config
from data import prepare, get_batch

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
tok, train_ds, val_ds, *_ = prepare(block_size=64)

def run_steps(cfg, steps=150, seed=0):
    torch.manual_seed(seed)
    m = M.GPT(cfg).to(device)
    opt = m.configure_optimizers(0.1, 3e-3, (0.9, 0.95), device_type=device)
    m.train()
    for s in range(steps):
        for g in opt.param_groups:
            g["lr"] = 3e-3 * min(1.0, (s + 1) / 20)
        x, y = get_batch(train_ds, 32, device)
        _, loss = m(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
    m.eval()
    with torch.no_grad():
        vx, vy = get_batch(val_ds, 32, device)
        _, vloss = m(vx, vy)
    return loss.item(), vloss.item(), sum(p.numel() for p in m.parameters())

def make(**over):
    cfg = get_config("micro")
    cfg.vocab_size = tok.vocab_size
    for k, v in over.items():
        setattr(cfg, k, v)
    cfg.__post_init__()
    return cfg


## 实验 1：基线 + 四组消融（固定随机种子、数据、步数）

In [4]:
import torch.nn as nn

# 让 MLP 可在 SwiGLU / relu^2 之间切换
class ReluSqMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = cfg.multiple_of * ((int(8 * cfg.n_embd / 3) + cfg.multiple_of - 1) // cfg.multiple_of)
        self.w_gate = nn.Linear(cfg.n_embd, h, bias=False)
        self.w_down = nn.Linear(h, cfg.n_embd, bias=False)
    def forward(self, x):
        return self.w_down(torch.relu(self.w_gate(x)).square())

results = []
results.append(("baseline (GQA,RoPE,SwiGLU,tied)", *run_steps(make(), 150)))

# 1) 关掉 GQA：n_kv_head = n_head
results.append(("no-GQA (MHA)", *run_steps(make(n_kv_head=4), 150)))
# 2) 关掉 RoPE：用可学习绝对位置
results.append(("no-RoPE (learned pos)", *run_steps(make(use_rope=False), 150)))
# 3) 不共享权重
results.append(("untied lm_head", *run_steps(make(tie_weights=False), 150)))

# 4) relu^2 替换 SwiGLU（monkeypatch 模块全局），跑完恢复
_orig_swiglu = M.SwiGLU
M.SwiGLU = ReluSqMLP
results.append(("relu^2 MLP", *run_steps(make(), 150)))
M.SwiGLU = _orig_swiglu


## 实验 2：结果表格

In [5]:
import importlib, model as M
importlib.reload(M)   # 恢复真实 SwiGLU

print(f"{'variant':32s} {'train':>8s} {'val':>8s} {'params(M)':>10s}")
for name, tr, va, p in results:
    print(f"{name:32s} {tr:8.4f} {va:8.4f} {p/1e6:10.4f}")

base = results[0][2]
print("\n相对基线的 val loss 变化：")
for name, tr, va, p in results[1:]:
    print(f"  {name:32s} {va - base:+.4f}")


variant                             train      val  params(M)
baseline (GQA,RoPE,SwiGLU,tied)    0.1457   4.4743     0.1020
no-GQA (MHA)                       0.1488   4.7751     0.1101
no-RoPE (learned pos)              0.4451   4.7433     0.1060
untied lm_head                     0.1222   5.0940     0.1055
relu^2 MLP                         0.1512   4.4183     0.0774

相对基线的 val loss 变化：
  no-GQA (MHA)                     +0.3008
  no-RoPE (learned pos)            +0.2690
  untied lm_head                   +0.6197
  relu^2 MLP                       -0.0560


## 分析提示

- **GQA**：把 KV 头从 4 减到 2，参数变少、KV cache 减半；短训练下 val loss 变化通常很小，
  说明「多个 query 头共享 KV」信息损失有限。
- **RoPE**：换成可学习绝对位置后，短序列上两者差距不大；但 RoPE 无参数、可外推。
- **权重共享**：`tie_weights=False` 多一份 $V\times C$ 参数，micro 上影响明显（vocab 小）。
- **MLP 激活**：`relu^2` 与 `SwiGLU` 在极小模型/短训练下接近，规模变大后 SwiGLU 常更优。

## 思考题
1. 为什么消融必须固定种子、数据、步数、优化器？只改一个变量意味着什么？
2. GQA 减少的主要是**参数**还是**计算**？（提示：KV 头复制后计算量如何变）
3. 若把步数从 150 增到 2000，各消融的差距会变大还是变小？为什么？

## 改动实验（任选 2–3 个）
1. 把 `n_layer` 从 2 增到 4，重做基线，看容量对 loss 的影响。
2. 加一组 `head_dim` 减半（`n_head=8`）的消融，观察是否变好/变差。
3. 把 `qk_norm` 关掉（改 `Block(cfg, use_qk_norm=False)` 需要小改代码），比较稳定性。

## 预期输出
```
variant                          train      val  params(M)
baseline (GQA,RoPE,SwiGLU,tied)  ~1.3     ~1.4     0.1030
no-GQA (MHA)                     ...       ...      0.1155
no-RoPE (learned pos)            ...       ...      0.1300
untied lm_head                   ...       ...      0.1270
relu^2 MLP                       ...       ...      0.1030
```
绝对数值依机器/种子略有波动，重要的是**同一运行内的相对比较**。
